In [ ]:
!pip install nltk rouge-score
!pip install bert-score

# Install TensorFlow for macOS (uncomment the appropriate line for your Mac)

# For Apple Silicon Macs (M1/M2/M3)
!pip install tensorflow-macos tensorflow-metal

# For Intel-based Macs, or fallback option
# !pip install tensorflow==2.10.0

# If the above doesn't work, try a lower version known to work on macOS
# !pip install tensorflow==2.8.0

/usr/local/bin/python3
3.11.3 (v3.11.3:f3909b8bc8, Apr  4 2023, 20:12:10) [Clang 13.0.0 (clang-1300.0.29.30)]


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [9]:
# Check if BLEURT-20 directory already exists before downloading
import os

if not os.path.exists('BLEURT-20'):
    print("BLEURT-20 directory not found. Downloading model...")
    # Use curl (Mac OS)
    !curl -L https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip -o BLEURT-20.zip
    !unzip -o BLEURT-20.zip
    print("BLEURT-20 model downloaded and extracted.")
else:
    print("BLEURT-20 directory already exists. Skipping download.")

BLEURT-20 directory already exists. Skipping download.


In [10]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

nltk.download('wordnet')
nltk.download('punkt')

def compute_bleu(reference, candidate):
    """
    Compute BLEU score for BLEU-1, BLEU-2, BLEU-3, and BLEU-4 between reference and candidate.
    Uses a smoothing function for short sentences.
    """
    reference_tokens = [reference.split()]
    candidate_tokens = candidate.split()
    smoothie = SmoothingFunction().method1  # Smoothing for short sequences
    
    bleu_scores = {}
    for n in range(1, 5):
        weights = tuple([1.0 / n] * n + [0.0] * (4 - n))
        bleu_scores[f"BLEU-{n}"] = sentence_bleu(reference_tokens, candidate_tokens, weights=weights, smoothing_function=smoothie)
    
    return bleu_scores

def compute_rouge(reference, candidate):
    """
    Compute ROUGE-L, ROUGE-1, and ROUGE-2 scores.
    Returns the F1 scores.
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, candidate)
    return {k: v.fmeasure for k, v in scores.items()}

def compute_meteor(reference, candidate):
    """
    Compute METEOR score.
    """
    reference_tokens = [reference.split()]
    candidate_tokens = candidate.split()
    return meteor_score(reference_tokens, candidate_tokens)

import bert_score.score
def compute_bertscore(reference, candidate, lang='en', model_type='bert-base-uncased'):
    P, R, F1 = bert_score.score([candidate], [reference], lang=lang, model_type=model_type, verbose=False)

    return {
        'precision': P[0].item(),
        'recall': R[0].item(),
        'f1': F1[0].item()
    }

# BLEURT score calculation function
try:
    import bleurt.score
    
    def compute_bleurt(references, candidates, checkpoint_path="BLEURT-20"):
        scorer = bleurt.score.BleurtScorer(checkpoint_path)
        scores = scorer.score(references=references, candidates=candidates)
        return scores
    
    print("BLEURT import successful!")
except ImportError as e:
    print(f"Warning: BLEURT import failed: {e}")
    print("Will use placeholder function for BLEURT")
    
    def compute_bleurt(references, candidates, checkpoint_path="BLEURT-20"):
        print("BLEURT unavailable, returning zero scores")
        return [0.0] * len(references)

[nltk_data] Error loading wordnet: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1002)>


BLEURT import successful!


[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1002)>


In [11]:
# Check if TensorFlow is available and print version
import sys
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

try:
    import tensorflow as tf
    print(f"TensorFlow version: {tf.__version__}")
    print(f"TensorFlow location: {tf.__file__}")
    print("TensorFlow is successfully installed and imported!")
    tf_available = True
except ImportError as e:
    print(f"ImportError: {e}")
    print("TensorFlow could not be imported. BLEURT scores will be set to zero.")
    tf_available = False

Python version: 3.11.3 (v3.11.3:f3909b8bc8, Apr  4 2023, 20:12:10) [Clang 13.0.0 (clang-1300.0.29.30)]
Python executable: /usr/local/bin/python3
TensorFlow version: 2.16.2
TensorFlow location: /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tensorflow/__init__.py
TensorFlow is successfully installed and imported!


In [12]:
# BLEURT score calculation function - improved detection logic
global_use_bleurt = False  # Will be set to True if BLEURT is successfully imported

try:
    # First check if TensorFlow is available
    import tensorflow as tf
    print(f"Using TensorFlow {tf.__version__}")
    
    # Then try to import BLEURT
    import bleurt.score
    
    def compute_bleurt(references, candidates, checkpoint_path="BLEURT-20"):
        """Compute BLEURT scores between references and candidates."""
        print(f"Initializing BLEURT scorer with checkpoint: {checkpoint_path}")
        scorer = bleurt.score.BleurtScorer(checkpoint_path)
        print(f"Computing BLEURT scores for {len(references)} examples...")
        scores = scorer.score(references=references, candidates=candidates)
        print(f"Successfully computed {len(scores)} BLEURT scores")
        return scores
    
    print("BLEURT import successful! Will compute BLEURT scores.")
    global_use_bleurt = True
    
except ImportError as e:
    print(f"Warning: Cannot use BLEURT: {e}")
    print("If you need BLEURT scores, make sure both TensorFlow and BLEURT are installed:")
    print("  pip install tensorflow-macos")
    print("  pip install git+https://github.com/google-research/bleurt.git")
    
    def compute_bleurt(references, candidates, checkpoint_path="BLEURT-20"):
        print("BLEURT unavailable, returning zero scores")
        return [0.0] * len(references)

Using TensorFlow 2.16.2
BLEURT import successful! Will compute BLEURT scores.


In [13]:
import pandas as pd
import os

results_folder = "results"
# result_filename = "results/ablation/pororo_ablation_language_gpt_4o_mini.csv"
result_filename = "results/ablation/pororo_ablation_visual_language_gpt_4o_mini.csv"
evaluation_results_folder = "results/evaluation"

os.makedirs(evaluation_results_folder, exist_ok=True)

print(f"Processing file: {result_filename}")
result = pd.read_csv(result_filename)

# Convert qid to integer if it's not already
if 'qid' in result.columns:
    result['qid'] = result['qid'].astype(str).str.replace('.0', '', regex=False)

os.makedirs(evaluation_results_folder, exist_ok=True)

for i, row in result.iterrows():
    if i == len(result)-1: # avoid the last row with total and average values
        continue

    reference_answer = row["correct_answer"].lower()
    generated_answer = row["predicted_answer"].lower()

    bleus = compute_bleu(reference_answer, generated_answer)
    rouges = compute_rouge(reference_answer, generated_answer)
    meteor = compute_meteor(reference_answer, generated_answer)
    berts = compute_bertscore(reference_answer, generated_answer)
    
    for n in range(1, 5):
        result.at[i, f"BLEU-{n}"] = bleus[f"BLEU-{n}"]
    
    for rouge_type, score_value in rouges.items():
        result.at[i, f"{rouge_type.upper()}"] = score_value
    
    result.at[i, "METEOR"] = meteor
    result.at[i, "BERTScore_Precision"] = berts["precision"]
    result.at[i, "BERTScore_Recall"] = berts["recall"]
    result.at[i, "BERTScore_F1"] = berts["f1"]

# Try to calculate BLEURT scores if TensorFlow and BLEURT are available
if global_use_bleurt:
    try:
        print("Calculating BLEURT scores...")
        reference_answers = list(result["correct_answer"])[:-1]
        generated_answers = list(result["predicted_answer"])[:-1]
        bleurts = compute_bleurt(reference_answers, generated_answers)
        result["BLEURT"] = bleurts + [""]
        print("Successfully calculated BLEURT scores")
    except Exception as e:
        print(f"BLEURT calculation failed: {e}")
        print("Skipping BLEURT score calculation")
        # Add zeros for BLEURT scores as fallback
        result["BLEURT"] = [0.0] * (len(result)-1) + [""]
else:
    print("Skipping BLEURT calculation because TensorFlow or BLEURT module is not available")
    # Add zeros for BLEURT scores
    result["BLEURT"] = [0.0] * (len(result)-1) + [""]

# Save results with 'metrics_' prefix
output_filename = "metrics_" + os.path.basename(result_filename)
output_path = os.path.join(evaluation_results_folder, output_filename)

result.to_csv(output_path, index=False)
print(f"Results saved to: {output_path}")

Processing file: results/ablation/pororo_ablation_visual_language_gpt_4o_mini.csv


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning:

Calculating BLEURT scores...
Initializing BLEURT scorer with checkpoint: BLEURT-20
INFO:tensorflow:Reading checkpoint BLEURT-20.


INFO:tensorflow:Reading checkpoint BLEURT-20.


INFO:tensorflow:Config file found, reading.


...

INFO:tensorflow:Loading model.


INFO:tensorflow:Loading model.


INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:BLEURT initialized.


Computing BLEURT scores for 40 examples...
Successfully computed 40 BLEURT scores
Successfully calculated BLEURT scores
Results saved to: results/evaluation/metrics_pororo_ablation_visual_language_gpt_4o_mini.csv
Successfully computed 40 BLEURT scores
Successfully calculated BLEURT scores
Results saved to: results/evaluation/metrics_pororo_ablation_visual_language_gpt_4o_mini.csv


In [14]:
result

,row_num,video_name,gif_num,qid,question,correct_answer,predicted_answer,evaluator_scores,accuracy,BLEU-1,...,BLEU-3,BLEU-4,ROUGE1,ROUGE2,ROUGEL,METEOR,BERTScore_Precision,BERTScore_Recall,BERTScore_F1,BLEURT
0,1,Pororo_ENGLISH1_1_ep1,14,383,whtat does eddy ask pororo,he asks pororo what are you doing,eddy asks pororo about the reason for his urge...,"0.25,0.25,0.25",0.2500,0.200000,...,0.065248,0.044632,0.235294,0.133333,0.235294,0.256849,0.560110,0.595777,0.577394,0.500187
1,2,Pororo_ENGLISH1_1_ep10,12,1100,were eddy's friends interested seeing his new ...,"yes, they ran happily towards the new toy.","yes, eddy's friends are interested in seeing h...","1.0,1.0,1.0",1.0000,0.117647,...,0.016987,0.013679,0.230769,0.083333,0.230769,0.112360,0.515543,0.646743,0.573738,0.517221
2,3,Pororo_ENGLISH1_1_ep10,4,1090,what did eddy say after getting the book?,"eddy told, "" what should i make today""","eddy joyfully exclaimed, ""ah, i found it","0.25,0.0,0.25",0.2500,0.247679,...,0.039588,0.034052,0.285714,0.000000,0.285714,0.126582,0.539352,0.534660,0.536996,0.267983
3,4,Pororo_ENGLISH1_1_ep11,51,1181,why did pororo look to ground?,because he was sorry.,"pororo looks to the ground, possibly reflectin...","0.75,0.75,0.75",0.7500,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.264435,0.330734,0.293892,0.157577
4,5,Pororo_ENGLISH1_1_ep12,29,1215,what exploded in pororo's face,a bomb box exploded pororo's face,a bomb box that crong hid exploded in pororo's...,"1.0,1.0,1.0",1.0000,0.357143,...,0.166054,0.080323,0.636364,0.400000,0.636364,0.655882,0.640592,0.841513,0.727434,0.562906
5,6,Pororo_ENGLISH1_1_ep12,36,1222,what does poby ask when he sees eddy,poby asks eddy why is he so jumpy,poby likely asks eddy about their surroundings...,"0.25,0.25,0.25",0.2500,0.214286,...,0.051597,0.033429,0.272727,0.100000,0.272727,0.297158,0.483865,0.567137,0.522202,0.39439
6,7,Pororo_ENGLISH1_1_ep12,43,1226,what does confess in loopy's house,eddy confesses he placed the box in pororo's h...,"in loopy's house, eddy confesses to placing th...","1.0,1.0,1.0",1.0000,0.238095,...,0.050042,0.028886,0.500000,0.266667,0.312500,0.418028,0.519190,0.732387,0.607630,0.469853
7,8,Pororo_ENGLISH1_1_ep12,49,1232,what does pororo say to crong after he realize...,pororo apologizes to crong and says he made a ...,pororo expresses his realization to crong by s...,"0.25,0.25,0.25",0.2500,0.160000,...,0.030718,0.019052,0.285714,0.060606,0.285714,0.173913,0.534193,0.688308,0.601536,0.472184
8,9,Pororo_ENGLISH1_1_ep13,12,1258,did eddy stay longer after agreeing to sing,"no, he left right away",eddy did not stay longer after agreeing to sin...,"1.0,1.0,1.0",1.0000,0.055556,...,0.012688,0.010802,0.086957,0.000000,0.086957,0.079365,0.346496,0.401966,0.372176,0.299135
9,10,Pororo_ENGLISH1_1_ep13,41,1283,did eddy's entrance impress the audience,"yes, they were all surprised and clapped","eddy's entrance likely impressed the audience,...","1.0,1.0,1.0",1.0000,0.055556,...,0.012688,0.010802,0.153846,0.083333,0.153846,0.123457,0.368368,0.448109,0.404345,0.382176
